In [ ]:
import eos
import os
import numpy as np
import matplotlib.pyplot as plt
import ast
import cmath
from wquantiles import quantile_1D
from scipy.stats import chi2
import pandas as pd
from IPython.display import display


colors = eos.figure.ItemColorCycler()._colors

resample = False

In [ ]:
BASE_DIRECTORY = os.getcwd()
if not os.path.exists(BASE_DIRECTORY):
    os.makedirs(os.path.abspath(BASE_DIRECTORY), exist_ok=True)

In [ ]:
def wstd(samples, weights=None, axis=0):
    mean_w = np.average(samples, weights=weights, axis=axis)
    return np.sqrt(np.average((samples - mean_w)**2, weights=weights, axis=axis))

def get_gof(analysis, param_array):
    parameter_list = []

    for param in analysis.init_args['priors']:
        if param['type'] == 'transform':
            parameter_list += param['parameters']
        else:
            parameter_list.append(param['parameter'])

    for i, param in enumerate(parameter_list):
        analysis.parameters.set(param, param_array[i])

    return analysis.goodness_of_fit()

In [ ]:
colors = eos.figure.ItemColorCycler()._colors

experiment_colors = {
    "Belle": colors[0],
    "CLEO": colors[1],
    "ALEPH": colors[2],
    "OPAL": colors[4],
}

EXPERIMENTS = tuple(experiment_colors)

fit_results = {}

In [ ]:
def prepare_fit(order, experiment, resample, refit=False):
    posterior = f"FF-fp-I1-order{order}-{experiment}"
    analysis = af.analysis(posterior)

    fit_directory = os.path.join(
        BASE_DIRECTORY, "data", posterior
    )

    # Generate new samples only when requested.
    if resample:
        eos.tasks.sample_nested(
            af,
            posterior,
            base_directory=BASE_DIRECTORY,
            bound="multi",
            nlive=200,
        )

    f = eos.data.ImportanceSamples(
        os.path.join(fit_directory, "samples")
    )

    mode_path = os.path.join(fit_directory, "mode-default")

    # Refit when requested, after resampling, or if no mode is saved.
    if refit or resample or not os.path.exists(mode_path):
        bfp, gof = eos.tasks.find_mode(
            af,
            posterior,
            importance_samples=True,
            base_directory=BASE_DIRECTORY,
            optimizations=100,
        )
    else:
        mode = eos.data.Mode(mode_path)
        gof = get_gof(analysis, mode.mode)
        bfp = eos.BestFitPoint(analysis, mode.mode)

    return {
        "posterior": posterior,
        "analysis": analysis,
        "f": f,
        "bfp": bfp,
        "gof": gof,
    }

In [ ]:
def show_fit_rotation(result):
    fit = result["f"]
    samples = fit.samples
    weights = fit.weights
    size = samples.shape[1]

    parameter_names = [
        parameter.name()
        for parameter in result["analysis"].varied_parameters
    ]

    cov = np.atleast_2d(
        np.cov(samples, rowvar=False, aweights=weights)
    )

    if not np.isfinite(cov).all():
        raise ValueError("The weighted covariance contains non-finite values.")

    try:
        chol = np.linalg.cholesky(np.linalg.inv(cov))
    except np.linalg.LinAlgError as error:
        raise ValueError(
            "Cannot compute the rotation: the covariance must be "
            "positive definite. Check for constant or linearly dependent "
            "parameters, or too few independent samples."
        ) from error

    corr = cov / np.sqrt(np.outer(np.diag(cov), np.diag(cov)))

    # Same uncentred transformation as in your original code.
    decorrelated_samples = samples @ chol

    # For column vectors: original_parameters = transform @ z.
    # This is equivalent to np.linalg.inv(chol.T).
    transform = np.linalg.solve(chol.T, np.eye(size))

    means = np.average(
        decorrelated_samples, weights=weights, axis=0
    )
    stds = wstd(
        decorrelated_samples, weights=weights, axis=0
    )

    decorrelated_min = means - 5 * stds
    decorrelated_max = means + 5 * stds

    decorrelated_bfp = np.asarray(result["bfp"].point) @ chol

    print(f"{result['posterior']}: rotation diagnostics")
    print("Original parameter order:")
    for index, name in enumerate(parameter_names, start=1):
        print(f"p{index}: {name}")

    # Correlation matrix in the original basis.
    correlation_figure, ax = plt.subplots(figsize=(6, 5))

    image = ax.imshow(
        corr,
        vmin=-1,
        vmax=1,
        cmap="coolwarm",
    )

    labels = [f"p{i + 1}" for i in range(size)]
    ax.set_xticks(range(size), labels=labels)
    ax.set_yticks(range(size), labels=labels)
    ax.set_title("Weighted correlation matrix")

    correlation_figure.colorbar(image, ax=ax)
    correlation_figure.tight_layout()
    plt.show()

    print("Transformation back to original parameters:")
    display(transform)

    print("Shift: zero, matching your original convention.")
    print("Lower bounds in transformed coordinates:")
    display(decorrelated_min)

    print("Upper bounds in transformed coordinates:")
    display(decorrelated_max)

    print("Best-fit point in transformed coordinates:")
    display(decorrelated_bfp)

    # Prepare the bin edges once for each transformed coordinate.
    bins = [
        np.linspace(lower, upper, 100)
        for lower, upper in zip(decorrelated_min, decorrelated_max)
    ]

    corner_figure, axes = plt.subplots(
        size,
        size,
        figsize=(3.0 * size, 3.0 * size),
        dpi=100,
        squeeze=False,
    )

    for i in range(size):
        for j in range(size):
            ax = axes[i, j]

            if j < i:
                ax.set_axis_off()
                continue

            if i == j:
                ax.hist(
                    decorrelated_samples[:, i],
                    weights=weights,
                    bins=bins[i],
                    density=True,
                    alpha=0.5,
                    color="C1",
                )
            else:
                ax.hist2d(
                    decorrelated_samples[:, j],
                    decorrelated_samples[:, i],
                    weights=weights,
                    bins=[bins[j], bins[i]],
                    cmap="Greys",
                    rasterized=True,
                )

                ax.plot(
                    decorrelated_bfp[j],
                    decorrelated_bfp[i],
                    marker="*",
                    linestyle="none",
                    markersize=12,
                    color="tab:red",
                )

                ax.set_ylim(decorrelated_min[i], decorrelated_max[i])
                ax.set_ylabel(rf"$z_{{{i + 1}}}$")

            ax.set_xlim(decorrelated_min[j], decorrelated_max[j])
            ax.set_xlabel(rf"$z_{{{j + 1}}}$")
            ax.set_box_aspect(1)

    corner_figure.tight_layout()
    plt.show()

    result["rotation"] = {
        "parameters": parameter_names,
        "covariance": cov,
        "correlation": corr,
        "chol": chol,
        "transform": transform,
        "shift": np.zeros(size),
        "samples": decorrelated_samples,
        "min": decorrelated_min,
        "max": decorrelated_max,
        "bfp": decorrelated_bfp,
        "figures": {
            "correlation": correlation_figure,
            "corner": corner_figure,
        },
    }

    return result["rotation"]

In [ ]:
def show_fit(result, rotate=False):
    posterior = result["posterior"]
    analysis = result["analysis"]
    fit = result["f"]

    print(posterior)
    print(f"Collected {len(fit.samples)} samples")

    display(result["gof"])
    display(result["bfp"].point)

    means = np.average(
        fit.samples, weights=fit.weights, axis=0
    )
    stds = wstd(
        fit.samples, weights=fit.weights, axis=0
    )

    for parameter, mean, std in zip(
        analysis.varied_parameters, means, stds
    ):
        print(
            f"      - {{ 'parameter': '{parameter.name()}', "
            f"'min': {mean - 5*std:.5f}, "
            f"'max': {mean + 5*std:.5f}, "
            f"'type': 'uniform'}}"
        )

   # Construct the corner figure directly from the saved samples.
    figure_args = {
        'type': 'corner',
        'contents': [{
            'path': os.path.join(
                BASE_DIRECTORY, 'data', posterior, 'samples'
            ),
            'label': experiment,
            'color': experiment_colors[experiment],
            'kde': False,
        }],
        'variables': [
            entry['name'] for entry in fit.varied_parameters
        ],
    }

    figure = eos.figure.FigureFactory.from_dict(**figure_args)
    figure.draw()
    plt.show()

    result['fit_corner'] = figure

    if rotate:
        show_fit_rotation(result)

In [ ]:
def show_rho_mass_width(order):
    items = []

    for experiment in EXPERIMENTS:
        result = fit_results[order][experiment]
        f = result["f"]
        color = experiment_colors[experiment]

        # Columns: mass, width.
        rho_samples = f.samples[:, :2]

        means = np.average(
            rho_samples, weights=f.weights, axis=0
        )
        stds = wstd(
            rho_samples, weights=f.weights, axis=0
        )

        print(
            f"{experiment} rho mass  = "
            f"{means[0]:.4f} ± {stds[0]:.4f} GeV"
        )
        print(
            f"{experiment} rho width = "
            f"{means[1]:.4f} ± {stds[1]:.4f} GeV"
        )

        items.append({
            "type": "kde2D",
            "label": experiment,
            "color": color,
            "levels": [68, 95],
            "contours": ["lines", "labels"],
            "datafile": os.path.join(
                BASE_DIRECTORY,
                "data",
                result["posterior"],
                "samples",
            ),
            "variables": [
                "0->pipi::M_(+,1,0)@BHKMNR2026",
                "0->pipi::Gamma_(+,1,0)@BHKMNR2026",
            ],
        })

    items.append({
        "type": "errorbars",
        "positions": [[0.763, 0.145]],
        "xerrors": [0.002],
        "yerrors": [0.003],
        "label": "PDG 2026",
        "color": "black",
    })

    figure_args = {
        "watermark": {"position": "lower right"},
        "plot": {
            "xaxis": {
                "label": r"$M_\rho$",
                "unit": r"$\mathrm{GeV}$",
                "range": [0.75, 0.78],
            },
            "yaxis": {
                "label": r"$\Gamma_\rho$",
                "unit": r"$\mathrm{GeV}$",
                "range": [0.13, 0.17],
            },
            "legend": {"position": "lower right"},
            "items": items,
        },
    }

    figure = eos.figure.FigureFactory.from_dict(**figure_args)
    figure.draw()
    plt.show()

    return figure

In [ ]:
def show_dispersive_bound(order, recompute=False):
    prediction_name = "FF-fp-I1-dispersive-bound"

    for experiment in EXPERIMENTS:
        result = fit_results[order][experiment]
        posterior = result["posterior"]

        prediction_path = os.path.join(
            BASE_DIRECTORY,
            "data",
            posterior,
            f"pred-{prediction_name}",
        )

        # Compute missing predictions or update them when requested.
        if recompute or not os.path.exists(prediction_path):
            eos.tasks.predict_observables(
                af,
                posterior,
                prediction_name,
                base_directory=BASE_DIRECTORY,
            )

        preds = eos.data.Prediction(prediction_path)
        values = preds.samples[:, 0]

        mean = np.average(values, weights=preds.weights)
        std = wstd(values, weights=preds.weights)

        result["dispersive_bound"] = {
            "prediction": preds,
            "values": values,
            "mean": mean,
            "std": std,
        }

        print(
            f"{experiment} dispersive bound is "
            f"{mean:.4f} ± {std:.4f}"
        )

In [ ]:
def plot_dispersive_bound_histogram(order, bins):
    figure, ax = plt.subplots(figsize=(8, 5))

    for experiment in EXPERIMENTS:
        result = fit_results[order][experiment]["dispersive_bound"]

        ax.hist(
            result["values"],
            weights=result["prediction"].weights,
            bins=bins,
            histtype="step",
            linewidth=2,
            color=experiment_colors[experiment],
            label=experiment,
        )

    ax.set_xlabel("Dispersive bound")
    ax.set_ylabel("Sum of weights")
    ax.legend()

    figure.tight_layout()
    plt.show()

    return figure

In [ ]:
def get_ff_data_item(result, experiment):
    analysis = result["analysis"]

    constraint_names = list(analysis.init_args["likelihood"])
    constraint_names.extend(
        (analysis.init_args.get("manual_constraints") or {}).keys()
    )

    prefix = f"0->pipi::Abs{{f_+}}^2@{experiment}:"

    matches = sorted({
        str(name)
        for name in constraint_names
        if str(name).startswith(prefix)
    })

    if len(matches) != 1:
        raise ValueError(
            f"{experiment}: expected one constraint starting with "
            f"{prefix!r}, but found {matches}."
        )

    constraint_name = matches[0]
    entry = eos.Constraints()[constraint_name]

    observable_names = sorted({
        str(name)
        for name in entry.observables()
        if str(name).startswith("0->pipi::Abs{f_+}^2(")
    })

    if len(observable_names) != 1:
        raise ValueError(
            f"{constraint_name}: expected one form-factor observable, "
            f"but found {observable_names}."
        )

    year = constraint_name.split("@", 1)[1].split(":", 1)[1][:4]

    return {
        "type": "constraint",
        "label": f"{experiment} {year}",
        "color": experiment_colors[experiment],
        "variable": "q2",
        "observable": observable_names[0],
        "constraints": constraint_name,
    }

In [ ]:
def plot_form_factor_squared(order, experiment, recompute=False):
    result = fit_results[order][experiment]
    posterior = result["posterior"]

    data_item = get_ff_data_item(result, experiment)

    prediction_name = "FF-fp2-I1-all"
    prediction_path = os.path.join(
        BASE_DIRECTORY,
        "data",
        posterior,
        f"pred-{prediction_name}",
    )

    # Both plots use this same prediction file.
    if recompute or not os.path.exists(prediction_path):
        eos.tasks.predict_observables(
            af,
            posterior,
            prediction_name,
            base_directory=BASE_DIRECTORY,
        )

    prediction_item = {
        "type": "uncertainty",
        "label": f"N = {order} fit",
        "color": colors[5],
        "variable": "q2",
        "range": [-1.2, 2.0],
        "interpolation": "cubic",
        "datafile": prediction_path,
        "resolution": 300,
    }

    na7_item = {
        "type": "constraint",
        "label": "NA7 1986",
        "color": colors[3],
        "variable": "q2",
        "observable": "0->pipi::Abs{f_+}^2(q2)",
        "constraints": "0->pipi::Abs{f_+}^2@NA7:1986A",
    }

    views = {
        "full": {
            "xrange": [-1.2, 2.0],
            "yrange": [0.1, 100.0],
            "watermark": "lower right",
            "legend": "upper left",
        },
        "zoom": {
            "xrange": [0.4, 0.8],
            "yrange": [10.0, 60.0],
            "watermark": "upper right",
            "legend": "lower center",
        },
    }

    figures = {}

    for view_name, settings in views.items():
        items = [data_item.copy()]

        if view_name == "full":
            items.append(na7_item.copy())

        items.append(prediction_item.copy())

        if view_name == "full":
            # Shade the entire plot height above sIn.
            items.append({
                "type": "band",
                "color": "gray",
                "alpha": 0.25,
                "x": [sIn, 2.0],
            })

        figure_args = {
            "watermark": {
                "position": settings["watermark"],
            },
            "plot": {
                "xaxis": {
                    "label": r"$s$",
                    "unit": r"$\mathrm{GeV}^2$",
                    "range": settings["xrange"],
                },
                "yaxis": {
                    "label": r"$|F^{I=1}_\pi|^2$",
                    "range": settings["yrange"],
                    "scale": "log",
                },
                "legend": {
                    "position": settings["legend"],
                },
                "items": items,
            },
        }

        print(f"{experiment}, order {order}: {view_name}")

        figure = eos.figure.FigureFactory.from_dict(**figure_args)
        figure.draw()
        plt.show()

        figures[view_name] = figure

    result["form_factor_squared"] = {
        "prediction_path": prediction_path,
        "figures": figures,
    }

    return figures

In [ ]:
def plot_form_factor_residuals(order, experiment):
    result = fit_results[order][experiment]
    analysis = result["analysis"]

    # Copy all parameter values, including fixed parameters.
    param_dict = {
        parameter.name(): parameter.evaluate()
        for parameter in analysis.parameters
    }

    # Insert the saved best-fit values of the varied parameters.
    # This does not require another goodness-of-fit calculation.
    for parameter, value in zip(
        analysis.varied_parameters,
        result["bfp"].point,
    ):
        param_dict[parameter.name()] = float(value)

    # Use the model options from this fit.
    fit_options = dict(
        analysis.init_args.get("global_options") or {}
    )
    fit_options.setdefault("form-factors", "BHKMNR2026")

    # Reuse the constraint lookup from the previous plotting block.
    residual_item = get_ff_data_item(result, experiment)

    residual_item.update({
        "type": "constraint-residue",
        "parameters": param_dict,
        "options": fit_options,
        "style": "delta",
    })

    figure_args = {
        "watermark": {"position": "lower right"},
        "plot": {
            "xaxis": {
                "label": r"$s$",
                "unit": r"$\mathrm{GeV}^2$",
                "range": [0.0, sIn],
            },
            "yaxis": {
                "label": (
                    r"$|F_\pi|^2_{\mathrm{data}}"
                    r" - |F_\pi|^2_{\mathrm{fit}}$"
                ),
                "range": [-1.5, 2.0],
                "scale": "linear",
            },
            "legend": {"position": "upper left"},
            "items": [residual_item],
        },
    }

    print(f"{experiment}, order {order}: residuals at the BFP")

    figure = eos.figure.FigureFactory.from_dict(**figure_args)
    figure.draw()
    plt.show()

    result["form_factor_residuals"] = figure

    return figure

In [ ]:
def plot_psi_phase(order, experiment, resolution=200):
    result = fit_results[order][experiment]
    analysis = result["analysis"]

    # Copy all parameters, including fixed parameters.
    param_dict = {
        parameter.name(): parameter.evaluate()
        for parameter in analysis.parameters
    }

    # Insert the saved best-fit values.
    for parameter, value in zip(
        analysis.varied_parameters,
        result["bfp"].point,
    ):
        param_dict[parameter.name()] = float(value)

    fit_options = dict(
        analysis.init_args.get("global_options") or {}
    )
    fit_options.setdefault("form-factors", "BHKMNR2026")

    figure_args = {
        'plot': {
            'xaxis': {
                'label': r'$\mathrm{Re}(\psi)$',
                'range': [-1.1, 1.1],
            },
            'yaxis': {
                'label': r'$\mathrm{Im}(\psi)$',
                'range': [-1.1, 1.1],
            },
            'items': [
                {
                    'type': 'complex-plane',
                    'ranges': ((-1.1, 1.1), (-1.1, 1.1)),
                    'variables': ['Re{psi}', 'Im{psi}'],
                    'observable': (
                        '0->pipi::Arg{f_+}(Re{psi},Im{psi})'
                        '@BHKMNR2026'
                    ),
                    'fixed_parameters': param_dict,
                    'options': fit_options,
                    'resolution': resolution,
                },
                {
                    'type': 'point',
                    'x': -0.426913,
                    'y': 0.904292,
                    'marker': 'o',
                    'markersize': 10,
                    'color': 'black',
                    'alpha': 1.0,
                },
                {
                    'type': 'point',
                    'x': -0.426913,
                    'y': -0.904292,
                    'marker': 'o',
                    'markersize': 10,
                    'color': 'black',
                    'alpha': 1.0,
                },
            ],
        },
    }

    print(f"{experiment}, order {order}: phase in the psi plane")

    figure = eos.figure.FigureFactory.from_dict(**figure_args)
    figure.draw()

    ax = figure._ax

    # Apply the same phase color scale to every experiment.
    for artist in list(ax.images) + list(ax.collections):
        artist.set_cmap("hsv")
        artist.set_clim(-np.pi, np.pi)

    # Draw the unit circle.
    circle = plt.Circle(
        (0, 0),
        1.0,
        color="black",
        fill=False,
        linewidth=1.5,
        zorder=10,
    )
    ax.add_patch(circle)

    ax.set_aspect("equal", adjustable="box")

    plt.show()

    result["psi_phase"] = figure

    return figure

In [ ]:
def plot_timelike_phase(order, recompute=False):
    prediction_name = "FF-arg-fp-I1-timelike"

    # Load the CHS comparison data.
    q2_over_mpisquared, phase, err = np.loadtxt(
        os.path.join(BASE_DIRECTORY, "scripts", "d11-central.txt"),
        skiprows=1,
        unpack=True,
    )
    q2 = q2_over_mpisquared * mpi**2

    items = []

    for experiment in EXPERIMENTS:
        result = fit_results[order][experiment]
        posterior = result["posterior"]

        prediction_path = os.path.join(
            BASE_DIRECTORY,
            "data",
            posterior,
            f"pred-{prediction_name}",
        )

        # Reuse saved predictions unless recomputation is requested.
        if recompute or not os.path.exists(prediction_path):
            eos.tasks.predict_observables(
                af,
                posterior,
                prediction_name,
                base_directory=BASE_DIRECTORY,
            )

        items.append({
            "type": "uncertainty",
            "label": experiment,
            "color": experiment_colors[experiment],
            "interpolation": "cubic",
            "resolution": 250,
            "variable": "q2",
            "datafile": prediction_path,
        })

    items.extend([
        {
            "type": "band",
            "color": "gray",
            "alpha": 0.25,
            "x": [sIn, 2.5],
            "y": [0.0, 3.5],
        },
        {
            "type": "expression",
            "expression": "np.pi",
            "range": [0.0, 2.5],
            "color": "black",
        },
    ])

    figure_args = {
        "watermark": {"position": "lower right"},
        "plot": {
            "xaxis": {
                "label": r"$s$",
                "unit": r"$\mathrm{GeV}^2$",
                "range": [0.0, 2.5],
            },
            "yaxis": {
                "label": r"$\arg F^{I=1}_\pi$",
                "range": [0.0, 3.5],
            },
            "legend": {"position": "lower center"},
            "items": items,
        },
    }

    figure = eos.figure.FigureFactory.from_dict(**figure_args)
    figure.draw()
    ax = figure._ax

    # Preserve the experiment entries from the EOS legend.
    legend = ax.get_legend()
    handles = list(legend.legend_handles)
    labels = [text.get_text() for text in legend.get_texts()]

    chs_band = ax.fill_between(
        q2,
        phase - err,
        phase + err,
        label="CHS 2018",
        alpha=0.5,
        color=colors[5],
    )

    threshold_line = ax.axvline(
        x=sPlus,
        color="black",
        linestyle=":",
        linewidth=1.0,
        label=r"$s_+$",
    )

    ax.legend(
        handles=handles + [chs_band, threshold_line],
        labels=labels + ["CHS 2018", r"$s_+$"],
        loc="lower center",
    )

    plt.show()
    return figure

In [ ]:
mpi    = eos.Parameters()["mass::pi^+"].evaluate()
sPlus  = 4 * mpi**2
sMinus = 0.0
sIn    = 0.850509 #pi-omega threshold

In [ ]:
def a11_calculator(sl_params):
    r2, r3, _, _ = sl_params

    coefficient = (
        -0.36294823212268046 * r2
        +0.06946304407045559 * r3
    )

    return sPlus * coefficient.real


def b11_calculator(sl_params):
    r2, r3, r4, r5 = sl_params

    coefficient = (
         0.03311321577857029   * r2
        -0.23706596387010614   * r2**2
        -0.14455309077802547   * r3
        +0.04537099795088154   * r2*r3
        +0.07902198795670203   * r4
        -0.00453709979508815   * r5
    )

    return sPlus**2 * coefficient.real

def calculate_sl_values(sl_params):
    samples = np.asarray(sl_params.samples)

    if samples.ndim != 2 or samples.shape[1] != 4:
        raise ValueError(
            "Expected four sample columns in order: r2, r3, r4, r5."
        )

    # Give each calculator the four complete sample columns.
    sample_columns = samples.T

    a11_values = a11_calculator(sample_columns)
    b11_values = b11_calculator(sample_columns)

    return a11_values, b11_values

def summarize_sl_values(name, a11_values, b11_values, weights):
    a11_mean = np.average(a11_values, weights=weights)
    a11_std  = wstd(a11_values, weights=weights)

    b11_mean = np.average(b11_values, weights=weights)
    b11_std  = wstd(b11_values, weights=weights)

    print(f"{name} a11 = {a11_mean:.4f} ± {a11_std:.4f}")
    print(f"{name} b11 = {b11_mean:.4f} ± {b11_std:.4f}")

    return a11_mean, a11_std, b11_mean, b11_std

sl_parameters = eos.Parameters()

a11_parameter = sl_parameters.declare_and_insert(
    'derived::a11',
    r'a_{11}',
    eos.Unit.Unity(),
    0.0,
    -100.0,
    100.0
)

b11_parameter = sl_parameters.declare_and_insert(
    'derived::b11',
    r'b_{11}',
    eos.Unit.Unity(),
    0.0,
    -100.0,
    100.0
)


def save_sl_samples(posterior, a11_values, b11_values, weights):
    sl_values = np.column_stack((
        a11_values,
        b11_values
    ))

    output_path = os.path.join(
        BASE_DIRECTORY,
        'data',
        posterior,
        'scattering-lengths',
        'samples'
    )

    os.makedirs(
        os.path.dirname(output_path),
        exist_ok=True
    )

    eos.data.ImportanceSamples.create(
        output_path,
        [a11_parameter, b11_parameter],
        sl_values,
        weights
    )

    return output_path


def show_scattering_lengths(order, recompute=False):
    prediction_name = "FF-fp-I1-scattering-length-params"

    for experiment in EXPERIMENTS:
        result = fit_results[order][experiment]
        posterior = result["posterior"]

        prediction_path = os.path.join(
            BASE_DIRECTORY,
            "data",
            posterior,
            f"pred-{prediction_name}",
        )

        if recompute or not os.path.exists(prediction_path):
            eos.tasks.predict_observables(
                af,
                posterior,
                prediction_name,
                base_directory=BASE_DIRECTORY,
            )

        sl_params = eos.data.Prediction(prediction_path)

        # Calculate a11 and b11 for every sample before averaging.
        a11_values, b11_values = calculate_sl_values(sl_params)

        a11_mean, a11_std, b11_mean, b11_std = summarize_sl_values(
            experiment,
            a11_values,
            b11_values,
            sl_params.weights,
        )

        # Save paired samples with their original weights.
        samples_path = save_sl_samples(
            posterior,
            a11_values,
            b11_values,
            sl_params.weights,
        )

        result["scattering_lengths"] = {
            "prediction": sl_params,
            "a11_values": a11_values,
            "b11_values": b11_values,
            "a11_mean": a11_mean,
            "a11_std": a11_std,
            "b11_mean": b11_mean,
            "b11_std": b11_std,
            "samples_path": samples_path,
        }

In [ ]:
# Theory a11 and b11 plot
# Sample independent Gaussian literature values (these values are from table 2 of arxiv:hep-ph/013088v1).
rng = np.random.default_rng(42)
n_samples = 2000

a11_theory = rng.normal(0.0379, 0.0005, n_samples)
b11_theory = rng.normal(0.00567, 0.00013, n_samples)

# All sampled curves have equal weight.
theory_weights = np.full(n_samples, 1.0 / n_samples)

# Physical s values, from threshold to 0.2 GeV^2.
s_theory = np.linspace(sPlus, 0.2, 200)
eps = s_theory - sPlus

# Each row is one complete curve from one sampled (a11, b11) pair.
t_theory_samples = eps[None, :] / sPlus * ( a11_theory[:, None] + b11_theory[:, None] * eps[None, :] / sPlus)

# Save the parameter samples, curve samples, grid, and weights.
theory_path = os.path.join(BASE_DIRECTORY, 'data', 'theory', 't11-threshold-samples.npz')
os.makedirs(os.path.dirname(theory_path), exist_ok=True)

np.savez_compressed(theory_path, s=s_theory, a11=a11_theory, b11=b11_theory, t11=t_theory_samples, weights=theory_weights)

with np.load(theory_path) as saved:
    s_theory = saved["s"]
    t_theory_samples = saved["t11"]

# Pointwise central 68.27% band for your equally weighted theory samples.
t_lower, t_median, t_upper = np.quantile(
    t_theory_samples,
    [0.15865, 0.5, 0.84135],
    axis=0,
)

partial_wave_theory = {
    "s": s_theory,
    "lower": t_lower,
    "median": t_median,
    "upper": t_upper,
}

In [ ]:
def plot_partial_waves(order, experiment, theory_band, recompute=False):
    result = fit_results[order][experiment]
    posterior = result["posterior"]

    parts = ("real", "imag")
    prediction_paths = {}

    # Prepare the two predictions, shared by all three plots.
    for part in parts:
        prediction_name = f"{part}-partial-wave-I1"

        prediction_path = os.path.join(
            BASE_DIRECTORY,
            "data",
            posterior,
            f"pred-{prediction_name}",
        )

        if recompute or not os.path.exists(prediction_path):
            eos.tasks.predict_observables(
                af,
                posterior,
                prediction_name,
                base_directory=BASE_DIRECTORY,
            )

        prediction_paths[part] = prediction_path

    # Each entry contains: name, x range, y range, interpolation resolution.
    views = [
        ("full", [-2.5, 2.5], [-0.7, 1.1], 250),
        ("near_zero", [-sPlus, 0.1], [-0.5, 0.5], 3000),
        ("threshold", [0.0, 0.2], [-0.15, 0.15], 3000),
    ]

    figures = {}

    for view, x_range, y_range, resolution in views:
        items = [
            {
                "type": "uncertainty",
                "label": f"{part} part",
                "color": colors[index],
                "interpolation": "cubic",
                "resolution": resolution,
                "variable": "q2",
                "range": x_range,
                "datafile": prediction_paths[part],
            }
            for index, part in enumerate(parts)
        ]

        figure_args = {
            "watermark": {"position": "lower right"},
            "plot": {
                "xaxis": {
                    "label": r"$s$",
                    "unit": r"$\mathrm{GeV}^2$",
                    "range": x_range,
                },
                "yaxis": {
                    "label": rf"{experiment} $t_1^1$",
                    "range": y_range,
                },
                "legend": {"position": "upper left"},
                "items": items,
            },
        }

        print(f"{experiment}, order {order}: {view.replace('_', ' ')}")

        figure = eos.figure.FigureFactory.from_dict(**figure_args)
        figure.draw()
        ax = figure._ax

        if view != "full":
            # Preserve the real/imaginary entries from the EOS legend.
            legend = ax.get_legend()
            handles = list(legend.legend_handles)
            labels = [text.get_text() for text in legend.get_texts()]

            # Add the theory comparison only to the threshold plot.
            if view == "threshold":
                theory_handle = ax.fill_between(
                    theory_band["s"],
                    theory_band["lower"],
                    theory_band["upper"],
                    color="red",
                    alpha=0.3,
                    label="CGL 2001",
                )

                ax.plot(
                    theory_band["s"],
                    theory_band["median"],
                    color="red",
                    linewidth=1.5,
                )

                handles.append(theory_handle)
                labels.append("CGL 2001")

            # Reference lines for both zoomed plots.
            ax.axhline(0.0, color="black", linewidth=1.0)
            ax.axvline(0.0, color="black", linewidth=1.0)

            threshold_handle = ax.axvline(
                sPlus,
                color="black",
                linestyle=":",
                linewidth=1.0,
                label=r"$s_+$",
            )

            handles.append(threshold_handle)
            labels.append(r"$s_+$")

            ax.legend(
                handles=handles,
                labels=labels,
                loc="upper left",
            )

        figures[view] = figure
        plt.show()

    result["partial_waves"] = {
        "prediction_paths": prediction_paths,
        "figures": figures,
    }

    return figures

In [ ]:
def plot_scattering_lengths(
    order,
    xrange=(0.03, 0.1),
    yrange=(-0.02, 0.015),
):
    items = []

    for experiment in EXPERIMENTS:
        sl_result = fit_results[order][experiment]["scattering_lengths"]

        items.append({
            "type": "kde2D",
            "label": experiment,
            "color": experiment_colors[experiment],
            "levels": [68],
            "contours": ["lines"],
            "bandwidth": 1.0,
            "datafile": sl_result["samples_path"],
            "xrange": xrange,
            "yrange": yrange,
            "variables": ["derived::a11", "derived::b11"],
        })

    items.append({
        "type": "errorbars",
        "positions": [[0.0379, 0.00567]],
        "xerrors": [0.0005],
        "yerrors": [0.00013],
        "label": "CGL 2001",
        "color": "black",
    })

    figure_args = {
        "watermark": {"position": "lower left"},
        "plot": {
            "xaxis": {
                "label": r"$a_1^1$",
                "range": xrange,
            },
            "yaxis": {
                "label": r"$b_1^1$",
                "range": yrange,
            },
            "legend": {"position": "upper right"},
            "items": items,
        },
    }

    figure = eos.figure.FigureFactory.from_dict(**figure_args)
    figure.draw()
    plt.show()

    return figure

In [ ]:
def charge_radius_squared_calculator(fprime_values):
    # 6 * df/dpsi at psi(0) * dpsi/ds at s=0 * (hbar c)^2
    return 6 * fprime_values * (-1.31241754914090000000) * (0.19732697178003900000)**2 #fm^2

def show_charge_radius(order, recompute=False):
    prediction_name = "FF-fp-I1-pion-charge-radius-param"

    for experiment in EXPERIMENTS:
        result = fit_results[order][experiment]
        posterior = result["posterior"]

        prediction_path = os.path.join(
            BASE_DIRECTORY,
            "data",
            posterior,
            f"pred-{prediction_name}",
        )

        if recompute or not os.path.exists(prediction_path):
            eos.tasks.predict_observables(
                af,
                posterior,
                prediction_name,
                base_directory=BASE_DIRECTORY,
            )

        preds = eos.data.Prediction(prediction_path)

        # Calculate <r^2> for all samples at once.
        values = charge_radius_squared_calculator(
            preds.samples[:, 0]
        )

        mean = np.average(values, weights=preds.weights)
        std = wstd(values, weights=preds.weights)

        result["charge_radius_squared"] = {
            "prediction": preds,
            "values": values,
            "mean": mean,
            "std": std,
        }

        print(
            f"{experiment} <r^2> = "
            f"{mean:.4f} ± {std:.4f} fm^2"
        )

In [ ]:
def show_rpi(order):
    figure, ax = plt.subplots(figsize=(8, 4.5))

    for y, experiment in enumerate(EXPERIMENTS):
        posterior = f'FF-fp-I1-order{order}-{experiment}'
        prediction = eos.data.Prediction(
            os.path.join(
                BASE_DIRECTORY,
                'data',
                posterior,
                'pred-FF-fp-I1-pion-charge-radius-param',
            )
        )

        values = charge_radius_squared_calculator(
            prediction.samples[:, 0]
        )
        weights = prediction.weights

        lower = quantile_1D(values, weights, 0.16)
        median = quantile_1D(values, weights, 0.50)
        upper = quantile_1D(values, weights, 0.84)

        ax.errorbar(
            median,
            y,
            xerr=[[median - lower], [upper - median]],
            fmt='|',
            markersize=14,
            markeredgewidth=2,
            capsize=5,
            elinewidth=2,
            color=experiment_colors[experiment],
        )

    # Reference: Colangelo 2019.
    reference = 0.429
    reference_error = np.hypot(0.001, 0.004)
    reference_y = len(EXPERIMENTS)

    ax.errorbar(
        reference,
        reference_y,
        xerr=reference_error,
        fmt='|',
        markersize=14,
        markeredgewidth=2,
        capsize=5,
        elinewidth=2,
        color='black',
    )

    # Reference: PDG average
    pdg_y = len(EXPERIMENTS) + 1

    ax.errorbar(
        0.434281,
        pdg_y,
        xerr=0.005272,
        fmt='|',
        markersize=14,
        markeredgewidth=2,
        capsize=5,
        elinewidth=2,
        color='black',
    )

    ax.set_yticks(
        range(len(EXPERIMENTS) + 2),
        [
            *EXPERIMENTS,
            r'Colangelo',
            r'PDG average',
        ],
    )
    ax.invert_yaxis()
    ax.set_xlabel(r'$\langle r_\pi^2\rangle\;[\mathrm{fm}^2]$')
    ax.set_title(f'Pion charge radius squared, order {order}')
    ax.grid(axis='x', alpha=0.3)

    figure.tight_layout()
    plt.show()

In [ ]:
def get_f_rho_samples(samples, ana):
    parameters = ana.varied_parameters

    if samples.shape[1] != len(parameters):
        raise ValueError("Sample columns do not match the parameter list.")

    kinematics = eos.Kinematics({"Re{q2}": 0.0, "Im{q2}": 0.0})
    options = eos.Options({
        "form-factors": "BHKMNR2026",
        "n-resonances": "1",
    })

    # Create the observables once.
    obs_re = eos.Observable.make(
        "0->pipi::Re{f_+}(Re{q2},Im{q2})",
        ana.parameters, kinematics, options,
    )
    obs_im = eos.Observable.make(
        "0->pipi::Im{f_+}(Re{q2},Im{q2})",
        ana.parameters, kinematics, options,
    )

    values = np.empty((len(samples), 2))

    for index, sample in enumerate(samples):
        for parameter, value in zip(parameters, sample):
            parameter.set(float(value))

        s_rho = (sample[0] - 1j * sample[1] / 2)**2
        kinematics["Re{q2}"].set(float(s_rho.real))
        kinematics["Im{q2}"].set(float(s_rho.imag))

        values[index] = obs_re.evaluate(), obs_im.evaluate()

    return values

In [ ]:
def export_rho_residues_psi(order, recompute=False):
    for experiment in EXPERIMENTS:
        posterior = f"FF-fp-I1-order{order}-{experiment}"
        fit_directory = os.path.join(BASE_DIRECTORY, "data", posterior)
        fit = eos.data.ImportanceSamples(
            os.path.join(fit_directory, "samples")
        )

        residue_parts = []

        for part in ("re", "im"):
            prediction_name = f"{part}-rho-residue-psi"
            prediction_path = os.path.join(
                fit_directory, f"pred-{prediction_name}"
            )

            if recompute or not os.path.exists(prediction_path):
                eos.tasks.predict_observables(
                    af, posterior, prediction_name,
                    base_directory=BASE_DIRECTORY,
                )

            prediction = eos.data.Prediction(prediction_path)
            residue_parts.append(prediction.samples[:, 0])

        # Columns: Re(residue_psi), Im(residue_psi), M, Gamma.
        data = np.column_stack((
            residue_parts[0],
            residue_parts[1],
            fit.samples[:, :2],
        ))

        if not np.isfinite(data).all():
            raise ValueError(f"{experiment}: non-finite residue inputs.")

        body = ",\n".join(
            f"{{{r:.17g}, {i:.17g}, {m:.17g}, {g:.17g}}}"
            for r, i, m, g in data
        ).replace("e", "*^")

        output_path = os.path.join(
            BASE_DIRECTORY, f"residues_psi_{experiment}{order}.wl"
        )

        with open(output_path, "w", encoding="utf-8") as file:
            file.write("{\n" + body + "\n}")

        print(f"{experiment}: exported {len(data)} samples to {output_path}")

In [ ]:
COUPLING_COLUMNS = [
    ("derived::Re{g_rho_gamma}", r"$\mathrm{Re}\,g_{\rho\gamma}$"),
    ("derived::Im{g_rho_gamma}", r"$\mathrm{Im}\,g_{\rho\gamma}$"),
    ("derived::Re{g_rho_pi_pi}", r"$\mathrm{Re}\,g_{\rho\pi\pi}$"),
    ("derived::Im{g_rho_pi_pi}", r"$\mathrm{Im}\,g_{\rho\pi\pi}$"),
]

coupling_parameters = eos.Parameters()

for name, latex in COUPLING_COLUMNS:
    if not coupling_parameters.has(name):
        coupling_parameters.declare_and_insert(
            name, latex, eos.Unit.Unity(),
            0.0, -100.0, 100.0,
        )

coupling_parameter_list = [
    coupling_parameters[name] for name, _ in COUPLING_COLUMNS
]

def prepare_coupling_samples(order):
    saved_paths = {}

    for experiment in EXPERIMENTS:
        posterior = f"FF-fp-I1-order{order}-{experiment}"
        fit_directory = os.path.join(BASE_DIRECTORY, "data", posterior)

        fit = eos.data.ImportanceSamples(
            os.path.join(fit_directory, "samples")
        )

        residue_data = np.loadtxt(
            os.path.join(
                BASE_DIRECTORY, f"residues_s_{experiment}{order}.csv"
            ),
            delimiter=",",
            ndmin=2,
        )

        if residue_data.shape != (len(fit.samples), 2):
            raise ValueError(
                f"{experiment}: expected one real/imaginary residue "
                "pair per posterior sample."
            )

        res_s = residue_data[:, 0] + 1j * residue_data[:, 1]
        M = fit.samples[:, 0]
        Gamma = fit.samples[:, 1]
        s_rho = (M - 1j * Gamma / 2)**2

        print(f"{experiment}: evaluating couplings...", flush=True)

        analysis = af.analysis(posterior)
        f_components = get_f_rho_samples(fit.samples, analysis)
        F_I = f_components[:, 0] + 1j * f_components[:, 1]

        sigma = np.sqrt(1 - sPlus / s_rho)

        g_rho_gamma = np.sqrt(
            24j * np.pi * s_rho / (F_I * res_s * sigma**3)
        )
        g_rho_pi_pi = -(res_s / s_rho) * g_rho_gamma

        values = np.column_stack((
            g_rho_gamma.real,
            g_rho_gamma.imag,
            g_rho_pi_pi.real,
            g_rho_pi_pi.imag,
        ))

        if not np.isfinite(values).all():
            raise ValueError(f"{experiment}: non-finite coupling values.")

        # Record the sample ranges in the saved parameter descriptions.
        for parameter, column in zip(coupling_parameter_list, values.T):
            parameter.set_min(float(column.min()))
            parameter.set_max(float(column.max()))

        output_path = os.path.join(
            fit_directory, "rho-couplings", "samples"
        )
        os.makedirs(os.path.dirname(output_path), exist_ok=True)

        eos.data.ImportanceSamples.create(
            output_path,
            coupling_parameter_list,
            values,
            fit.weights,
        )

        saved_paths[experiment] = output_path
        print(f"{experiment}: saved {len(values)} coupling samples.")

    return saved_paths

In [ ]:
def show_rho_couplings(order):
    summaries = {}

    for experiment in EXPERIMENTS:
        posterior = f"FF-fp-I1-order{order}-{experiment}"

        saved = eos.data.ImportanceSamples(
            os.path.join(
                BASE_DIRECTORY, "data", posterior,
                "rho-couplings", "samples",
            )
        )

        means = np.average(
            saved.samples, weights=saved.weights, axis=0
        )
        stds = wstd(
            saved.samples, weights=saved.weights, axis=0
        )

        summaries[experiment] = {}

        for quantity, index in (
            ("g_rho_gamma", 0),
            ("g_rho_pi_pi", 2),
        ):
            summaries[experiment][quantity] = {
                "mean": means[index] + 1j * means[index + 1],
                "std_re": stds[index],
                "std_im": stds[index + 1],
            }

            print(
                f"{experiment}: {quantity} = "
                f"({means[index]:.4f} ± {stds[index]:.4f}) "
                f"+ i * ({means[index + 1]:.4f} ± {stds[index + 1]:.4f})"
            )

    return summaries

In [ ]:
def plot_rho_couplings_corner(order):
    figures = {}

    # Order: Re(g_gamma), Im(g_gamma), Re(g_pipi), Im(g_pipi).
    theory = np.array([5.01, -0.11, 5.99, -0.54])
    theory_errors = np.array([0.08, 0.09, 0.07, 0.12])
    size = len(theory)

    for experiment in EXPERIMENTS:
        samples_path = os.path.join(
            BASE_DIRECTORY,
            'data',
            f'FF-fp-I1-order{order}-{experiment}',
            'rho-couplings',
            'samples',
        )

        figure = eos.figure.CornerFigure.from_dict(
            contents=[{
                'path': samples_path,
                'label': experiment,
                'color': experiment_colors[experiment],
                'kde': False,
            }],
            variables=[name for name, _ in COUPLING_COLUMNS],
        )

        figure.prepare()
        grid = figure._figure

        for panel in grid.plots:
            for attribute in ('xaxis', 'yaxis'):
                axis = getattr(panel, attribute, None)
                if axis is not None and axis.label:
                    axis.label = '$' + axis.label.strip().strip('$') + '$'

        figure.draw()
        axes = grid._axes

        limits = []
        for i in range(size):
            lower, upper = axes[i * size + i].get_xlim()
            lower = min(lower, theory[i] - theory_errors[i])
            upper = max(upper, theory[i] + theory_errors[i])
            padding = 0.05 * (upper - lower)
            limits.append((lower - padding, upper + padding))

        theory_handle = None

        for i in range(size):
            axes[i * size + i].set_xlim(limits[i])

            for j in range(i):
                ax = axes[i * size + j]
                handle = ax.errorbar(
                    theory[j],
                    theory[i],
                    xerr=theory_errors[j],
                    yerr=theory_errors[i],
                    fmt='*',
                    color='tab:red',
                    markersize=12,
                    capsize=3,
                    zorder=10,
                )

                if theory_handle is None:
                    theory_handle = handle

                ax.set_xlim(limits[j])
                ax.set_ylim(limits[i])

        axes[0].figure.legend(
            [theory_handle],
            ['arXiv:2312.15015v2'],
            loc='upper right',
        )

        print(f'{experiment}, order {order}')
        plt.show()
        figures[experiment] = figure

    return figures

In [ ]:
RPI2_NAME = "derived::rpi_squared"


def prepare_radius_corner_samples(order):
    parameters = eos.Parameters()
    saved_paths = {}

    if not parameters.has(RPI2_NAME):
        parameters.declare_and_insert(
            RPI2_NAME,
            r"$\langle r_\pi^2\rangle\,[\mathrm{fm}^2]$",
            eos.Unit.Femtometer2(),
            0.0, -100.0, 100.0,
        )

    for experiment in EXPERIMENTS:
        posterior = f"FF-fp-I1-order{order}-{experiment}"
        fit_directory = os.path.join(BASE_DIRECTORY, "data", posterior)

        fit = eos.data.ImportanceSamples(
            os.path.join(fit_directory, "samples")
        )
        sl = eos.data.ImportanceSamples(
            os.path.join(fit_directory, "scattering-lengths", "samples")
        )
        couplings = eos.data.ImportanceSamples(
            os.path.join(fit_directory, "rho-couplings", "samples")
        )
        radius_prediction = eos.data.Prediction(
            os.path.join(
                fit_directory,
                "pred-FF-fp-I1-pion-charge-radius-param",
            )
        )

        # Check sample counts and relative weights before joining columns.
        for name, source in (
            ("charge radius", radius_prediction),
            ("scattering lengths", sl),
            ("couplings", couplings),
        ):
            if (
                len(source.samples) != len(fit.samples)
                or np.shape(source.weights) != np.shape(fit.weights)
            ):
                raise ValueError(
                    f"{experiment}: {name} samples do not match the fit."
                )

            if not np.allclose(
                source.weights / np.sum(source.weights),
                fit.weights / np.sum(fit.weights),
                rtol=1e-6,
                atol=1e-12,
            ):
                raise ValueError(
                    f"{experiment}: {name} weights do not match the fit."
                )

        radius_values = charge_radius_squared_calculator(
            radius_prediction.samples[:, 0]
        )

        values = np.column_stack((
            radius_values,
            fit.samples,
            sl.samples,
            couplings.samples,
        ))

        if not np.isfinite(values).all():
            raise ValueError(f"{experiment}: non-finite joint samples.")

        names = [RPI2_NAME]
        for source in (fit, sl, couplings):
            names.extend(
                entry["name"] for entry in source.varied_parameters
            )

        parameter_list = [parameters[name] for name in names]

        for parameter, column in zip(parameter_list, values.T):
            parameter.set_min(float(column.min()))
            parameter.set_max(float(column.max()))

        output_path = os.path.join(
            fit_directory, "radius-correlations", "samples"
        )
        os.makedirs(os.path.dirname(output_path), exist_ok=True)

        eos.data.ImportanceSamples.create(
            output_path,
            parameter_list,
            values,
            fit.weights,
        )

        saved_paths[experiment] = output_path
        print(f"{experiment}: saved joint samples {values.shape}")

    return saved_paths

In [ ]:
def plot_radius_corners(order, group):
    if group not in ('fit', 'scattering', 'couplings'):
        raise ValueError("Choose 'fit', 'scattering', or 'couplings'.")

    pdg_r2 = 0.434281       # fm^2
    pdg_r2_error = 0.005272 # fm^2

    pdg_lower = pdg_r2 - pdg_r2_error
    pdg_upper = pdg_r2 + pdg_r2_error
    pdg_xerr = pdg_r2_error

    cgl = {
        1: (0.0379, 0.0005),    # a11
        2: (0.00567, 0.00013),  # b11
    }

    coupling_theory = {
        1: (5.01, 0.08),
        2: (-0.11, 0.09),
        3: (5.99, 0.07),
        4: (-0.54, 0.12),
    }

    figures = {}

    for experiment in EXPERIMENTS:
        posterior = f'FF-fp-I1-order{order}-{experiment}'
        fit_directory = os.path.join(BASE_DIRECTORY, 'data', posterior)

        if group == 'fit':
            fit = eos.data.ImportanceSamples(
                os.path.join(fit_directory, 'samples')
            )
            names = [entry['name'] for entry in fit.varied_parameters]
        elif group == 'scattering':
            names = ['derived::a11', 'derived::b11']
        else:
            names = [name for name, _ in COUPLING_COLUMNS]

        variables = [RPI2_NAME] + names

        figure = eos.figure.CornerFigure.from_dict(
            contents=[{
                'path': os.path.join(
                    fit_directory, 'radius-correlations', 'samples'
                ),
                'label': experiment,
                'color': experiment_colors[experiment],
                'kde': False,
            }],
            variables=variables,
        )

        figure.prepare()
        grid = figure._figure

        for panel in grid.plots:
            for attribute in ('xaxis', 'yaxis'):
                axis = getattr(panel, attribute, None)
                if axis is not None and axis.label:
                    axis.label = '$' + axis.label.strip().strip('$') + '$'

        figure.draw()

        size = len(variables)
        axes = grid._axes

        # Use matching limits wherever a variable appears.
        limits = [
            axes[i * size + i].get_xlim()
            for i in range(size)
        ]

        references = (
            cgl if group == 'scattering'
            else coupling_theory if group == 'couplings'
            else {}
        )

        for i in range(size):
            lower, upper = limits[i]

            if i == 0:
                lower = min(lower, pdg_lower)
                upper = max(upper, pdg_upper)
            elif i in references:
                value, error = references[i]
                lower = min(lower, value - error)
                upper = max(upper, value + error)

            padding = 0.05 * (upper - lower)
            limits[i] = (lower - padding, upper + padding)

        cgl_handle = None
        coupling_handle = None

        for i in range(size):
            for j in range(i + 1):
                ax = axes[i * size + j]

                if j == 0 and i > 0:
                    ax.axvspan(
                        pdg_lower,
                        pdg_upper,
                        color='tab:orange',
                        alpha=0.20,
                        zorder=8,
                    )
                    ax.axvline(
                        pdg_r2,
                        color='tab:orange',
                        linewidth=1,
                        zorder=9,
                    )

                if group == 'scattering' and i == 2 and j == 1:
                    cgl_handle = ax.errorbar(
                        cgl[1][0],
                        cgl[2][0],
                        xerr=cgl[1][1],
                        yerr=cgl[2][1],
                        fmt='o',
                        color='black',
                        markersize=6,
                        capsize=3,
                        zorder=10,
                    )

                if group == 'couplings' and i > j and j > 0:
                    handle = ax.errorbar(
                        coupling_theory[j][0],
                        coupling_theory[i][0],
                        xerr=coupling_theory[j][1],
                        yerr=coupling_theory[i][1],
                        fmt='*',
                        color='tab:red',
                        markersize=12,
                        capsize=3,
                        zorder=10,
                    )
                    if coupling_handle is None:
                        coupling_handle = handle

                ax.set_xlim(limits[j])
                if i > j:
                    ax.set_ylim(limits[i])

        # Put the PDG point on the one-dimensional radius panel.
        radius_ax = axes[0]
        pdg_y = 0.85 * radius_ax.get_ylim()[1]
        pdg_handle = radius_ax.errorbar(
            pdg_r2,
            pdg_y,
            xerr=pdg_xerr,
            fmt='o',
            color='tab:orange',
            markersize=6,
            capsize=3,
            zorder=10,
        )
        radius_ax.set_xlim(limits[0])

        handles = [pdg_handle]
        labels = [r'PDG $r_\pi^2$']

        if cgl_handle is not None:
            handles.append(cgl_handle)
            labels.append('CGL 2001')

        if coupling_handle is not None:
            handles.append(coupling_handle)
            labels.append('arXiv:2312.15015v2')

        axes[0].figure.legend(
            handles,
            labels,
            loc='upper right',
        )

        print(f'{experiment}, order {order}: radius + {group}')
        plt.show()
        figures[experiment] = figure

    return figures

In [ ]:
def show_results_table(order):
    def summary(samples, weights, column, digits=3, factor=1.0):
        values = samples[:, column]

        median = factor * quantile_1D(values, weights, 0.50)
        lower = factor * quantile_1D(values, weights, 0.16)
        upper = factor * quantile_1D(values, weights, 0.84)

        err_lo = median - lower
        err_hi = upper - median

        return (
            f'{median:.{digits}f} '
            f'(+{err_hi:.{digits}f}/-{err_lo:.{digits}f})'
        )

    coupling_names = [name for name, _ in COUPLING_COLUMNS]
    if len(coupling_names) != 4:
        raise ValueError('Expected four entries in COUPLING_COLUMNS.')

    coupling_labels = (
        'Re(g_rhogamma)',
        'Im(g_rhogamma)',
        'Re(g_rhopipi)',
        'Im(g_rhopipi)',
    )

    rows = []

    for experiment in EXPERIMENTS:
        result = fit_results[order][experiment]
        posterior = result['posterior']
        fit_directory = os.path.join(
            BASE_DIRECTORY, 'data', posterior
        )

        # Remove the manually defined root constraint, then evaluate
        # goodness of fit at the original saved best-fit point.
        analysis = result['analysis']
        reduced_args = dict(analysis.init_args)
        manual_constraints = dict(
            reduced_args.get('manual_constraints') or {}
        )

        root_name = '0->pipi::root_penalty'
        matches = [
            name for name in manual_constraints
            if str(name) == root_name
        ]

        if len(matches) != 1:
            raise ValueError(
                f'{experiment}: expected {root_name!r} among the manual '
                f'constraints; found {list(manual_constraints)}'
            )

        del manual_constraints[matches[0]]
        reduced_args['manual_constraints'] = manual_constraints
        reduced_analysis = eos.Analysis(**reduced_args)

        best_fit_values = {
            parameter.name(): value
            for parameter, value in zip(
                analysis.varied_parameters,
                result['bfp'].point,
            )
        }

        for parameter in reduced_analysis.varied_parameters:
            reduced_analysis.parameters.set(
                parameter.name(),
                best_fit_values[parameter.name()],
            )

        reduced_gof = reduced_analysis.goodness_of_fit()
        bestchi2_without_root = reduced_gof.total_chi_square()

        # Your specified convention: root-penalty contributes one d.o.f.
        dof_without_root = (
            result['gof'].total_degrees_of_freedom() - 1
        )
        if dof_without_root <= 0:
            raise ValueError(
                f'{experiment}: dof after removing root-penalty '
                'must be positive.'
            )

        pvalue = 100 * chi2.sf(
            bestchi2_without_root,
            dof_without_root,
        )

        # Created earlier by prepare_radius_corner_samples(order).
        joint = eos.data.ImportanceSamples(
            os.path.join(
                fit_directory,
                'radius-correlations',
                'samples',
            )
        )

        def joint_summary(name, digits=3, factor=1.0):
            return summary(
                joint.samples,
                joint.weights,
                joint.lookup_table[name],
                digits=digits,
                factor=factor,
            )

        bound = eos.data.Prediction(
            os.path.join(
                fit_directory,
                'pred-FF-fp-I1-dispersive-bound',
            )
        )

        row = {
            'Experiment': experiment,
            'p-value (%)': f'{pvalue:.2f}',
            'M_rho (MeV)': joint_summary(
                '0->pipi::M_(+,1,0)@BHKMNR2026',
                digits=2,
                factor=1000,
            ),
            'Gamma_rho (MeV)': joint_summary(
                '0->pipi::Gamma_(+,1,0)@BHKMNR2026',
                digits=2,
                factor=1000,
            ),
            'Dispersive bound': summary(
                bound.samples,
                bound.weights,
                0,
                digits=4,
            ),
            'a11': joint_summary('derived::a11', digits=4),
            'b11': joint_summary('derived::b11', digits=4),
            'r_pi^2 (fm^2)': joint_summary(
                RPI2_NAME,
                digits=4,
            ),
        }

        for label, name in zip(coupling_labels, coupling_names):
            row[label] = joint_summary(name, digits=3)

        rows.append(row)

    table = pd.DataFrame(rows).set_index('Experiment')
    return table